<a href="https://colab.research.google.com/github/wonzzae/WJ-Archive/blob/main/fastapi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from fastapi import FastAPI, Query
from pydantic import BaseModel

app = FastAPI()

1. 첫걸음

In [2]:
@app.get("/")
def read_root():
    return {"Hello": "World"}

2. 경로 매개변수

In [3]:
@app.get("/items/{item_id}")
def read_item(item_id: int):
    return {"item_id": item_id}

3. Enum (선택지 제한)

In [4]:
from enum import Enum

class ModelName(str, Enum):
    alexnet = "alexnet"
    resnet = "resnet"
    lenet = "lenet"

@app.get("/models/{model_name}")
def get_model(model_name: ModelName):
    return {"model_name": model_name}

4. 쿼리 매개변수

In [5]:
@app.get("/items/")
def read_item(q: str = None):
    return {"q": q}

5. 쿼리 + 기본값

In [6]:
@app.get("/items2/")
def read_item2(q: str = "default"):
    return {"q": q}

쿼리 검증

In [7]:
@app.get("/items3/")
def read_item3(q: str = Query(None, min_length=3, max_length=10)):
    return {"q": q}

Path 검증

In [8]:
from fastapi import Path

@app.get("/items4/{item_id}")
def read_item4(item_id: int = Path(..., gt=0, lt=100)):
    return {"item_id": item_id}

요청 본문

In [9]:
class Item(BaseModel):
    name: str
    price: float
    description: str = None

@app.post("/items/")
def create_item(item: Item):
    return item

Path + Body 같이

In [10]:
@app.put("/items/{item_id}")
def update_item(item_id: int, item: Item):
    return {"item_id": item_id, "item": item}

10. 여러 Body

In [11]:
class User(BaseModel):
    username: str

@app.post("/items-with-user/")
def create_item_with_user(item: Item, user: User):
    return {"item": item, "user": user}

11. Response 모델

In [12]:
class ItemResponse(BaseModel):
    name: str
    price: float

@app.post("/response/", response_model=ItemResponse)
def create_response(item: Item):
    return item

12. 상태 코드

In [13]:
from fastapi import status

@app.post("/status/", status_code=status.HTTP_201_CREATED)
def create_status(item: Item):
    return item

13. 에러 처리

In [14]:
from fastapi import HTTPException

@app.get("/error/{item_id}")
def read_error(item_id: int):
    if item_id != 1:
        raise HTTPException(status_code=404, detail="Item not found")
    return {"item_id": item_id}

14. 헤더

In [15]:
from fastapi import Header

@app.get("/header/")
def read_header(user_agent: str = Header(None)):
    return {"User-Agent": user_agent}

15. 쿠키

In [16]:
from fastapi import Cookie

@app.get("/cookie/")
def read_cookie(session_id: str = Cookie(None)):
    return {"session_id": session_id}

In [18]:
!pip install pyngrok

In [21]:
import nest_asyncio
nest_asyncio.apply()

import uvicorn
from threading import Thread

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = Thread(target=run)
thread.start()

from pyngrok import ngrok
url = ngrok.connect(8000)
print(url)
print(url, "/docs")

Exception in thread Thread-5 (run):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_2261/2451268379.py", line 8, in run
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/main.py", line 617, in run
    server.run()
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/server.py", line 75, in run
    return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: _patch_asyncio.<locals>.run() got an unexpected keyword argument 'loop_factory'
ERROR:pyngrok.process.ngrok:t=2026-05-06T12:48:51+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.